## Objective B: Trip Duration & ETA Model

The goal of Objective B is to build a regression model that estimates expected travel time (ETA) based on bike type and trip characteristics, in order to better understand rider experience.


### !!! Note to whoever is reading this from the group. The steps here were also shown on DataCamp and they make much more sense (and are also the correct way to do it) compared to what we did in class.
### I did not just copy it from ChatGPT without researching (for Daniil) hahahha
### Feel free to edit however you want

### Spark Session Initialization
 Each notebook runs as an independent process, so the Spark session must be reinitialized before working with Spark data.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("CitiBike_TripDuration_Analysis") \
    .config("spark.executor.memory", "1gb") \
    .getOrCreate()

### Load the cleaned data from disk
Reloading the cleaned data avoids repeating expensive preprocessing steps and ensures a reproducible workflow.

In [2]:
df = spark.read.parquet("processed/clean_trips")

## 1. Exploratory analysis

In [3]:
df.select("trip_duration_sec").summary().show()

+-------+-----------------+
|summary|trip_duration_sec|
+-------+-----------------+
|  count|          4325553|
|   mean|803.3547953290597|
| stddev| 1775.10999075081|
|    min|               61|
|    25%|              318|
|    50%|              549|
|    75%|              955|
|    max|            90049|
+-------+-----------------+



### Trip duration by bike type

#### The first question we are interested in answering is: Do electric bikes differ from classic bikes in terms of trip duration?

In [4]:
df.groupBy("rideable_type") \
  .avg("trip_duration_sec") \
  .show()

+-------------+----------------------+
|rideable_type|avg(trip_duration_sec)|
+-------------+----------------------+
|electric_bike|     779.5942678097983|
| classic_bike|     857.6209295563574|
+-------------+----------------------+



#### How does user type affect trip duration?

In [5]:
df.groupBy("member_casual") \
  .avg("trip_duration_sec") \
  .show()

+-------------+----------------------+
|member_casual|avg(trip_duration_sec)|
+-------------+----------------------+
|       casual|    1265.7651729742831|
|       member|     691.6481652159562|
+-------------+----------------------+



#### Combining both gives us real insights

In [6]:
df.groupBy("rideable_type", "member_casual") \
  .avg("trip_duration_sec") \
  .show()

+-------------+-------------+----------------------+
|rideable_type|member_casual|avg(trip_duration_sec)|
+-------------+-------------+----------------------+
| classic_bike|       member|     705.3167622350625|
| classic_bike|       casual|     1590.196611619824|
|electric_bike|       casual|    1146.1483854600333|
|electric_bike|       member|     685.4204932498604|
+-------------+-------------+----------------------+



#### Answering these descriptive questions helps us understand patterns in the data before we start modeling

## 2. Building the model

Although Spark internally relies on RDDs, in our analysis we work with  DataFrames and Spark ML, which abstract away low-level RDD operations and provide a more optimized and user-friendly workflow.


Spark machine learning models cannot work directly with string values.  
`StringIndexer` converts categorical string columns (such as bike type or user type) into numerical indices, where each distinct category is mapped to an integer.   
This step is required before applying any machine learning model in Spark.


In [7]:
from pyspark.ml.feature import StringIndexer

bike_indexer = StringIndexer(
    inputCol="rideable_type",
    outputCol="rideable_type_idx"
)

user_indexer = StringIndexer(
    inputCol="member_casual",
    outputCol="member_casual_idx"
)

After indexing, categorical variables are represented as integers.  
However, these numbers do not have a natural order (e.g. electric bikes are not "greater" than classic bikes).   
`OneHotEncoder` converts indexed categories into binary vectors, ensuring that the regression model treats categories as nominal rather than ordinal.

In [8]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=["rideable_type_idx", "member_casual_idx"],
    outputCols=["rideable_type_vec", "member_casual_vec"]
)

Spark ML models expect all input features to be provided in a single vector column.  
`VectorAssembler` combines multiple feature columns into one unified feature vector that can be used by the regression model.
Rows containing missing feature values are automatically skipped during feature assembly to ensure robust model training.

In [14]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "rideable_type_vec",
        "member_casual_vec",
        "start_hour",
        "distance_km"
    ],
    outputCol="features",
    handleInvalid="skip"
)

Linear regression is used to estimate expected trip duration (ETA) as a continuous numerical value.  
The model learns the relationship between trip duration and selected trip characteristics, enabling interpretation of how each feature affects rider experience.


In [15]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="trip_duration_sec"
)

A Spark ML `pipeline` combines all preprocessing steps and the regression model into a single workflow.  
This ensures that feature transformations and model training are applied consistently and reproducibly.


In [16]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[
    bike_indexer,
    user_indexer,
    encoder,
    assembler,
    lr
])

To evaluate the regression model fairly, the dataset is split into training and test sets.  
The model is trained on the training data and evaluated on unseen test data to assess its predictive performance. 

In [17]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed = 123)

In [18]:
model = pipeline.fit(train_df)

In [19]:
predictions = model.transform(test_df)

To assess model performance, two standard regression metrics are used:
- **RMSE (Root Mean Squared Error)** measures the average prediction error in seconds.
- **R² (Coefficient of Determination)** indicates how much of the variation in trip duration is explained by the model.

These metrics are computed using Spark’s `RegressionEvaluator`

In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(
    labelCol="trip_duration_sec",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="trip_duration_sec",
    predictionCol="prediction",
    metricName="r2"
)

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("Test RMSE:", rmse)
print("Test R²:", r2)


Test RMSE: 1016.1554714531039
Test R²: 0.1730433008690574


### Interpretation of Results
With an RMSE of approximately 1,016 seconds (around 17 minutes), the model’s ETA predictions still show a noticeable deviation from the actual trip durations, indicating that individual trip times vary substantially. While the prediction error is still relatively large, this is expected given the wide variation in bike trips.

The R² value of approximately 0.17 indicates that the model explains about 17% of the variability in trip duration. This suggests that the included features capture some meaningful patterns related to rider experience, but a majority of the variation in travel time is influenced by factors not represented in the dataset, such as route choice, traffic conditions and individual riding behavior.

Overall, the model provides a rough estimate of trip duration but is not suitable for precise ETA prediction.
